# Lab 69: The Motion Reader

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-69.ipynb)

**What you will do:** test how accurately you can identify gender or emotion from point-light walker clips alone, compare your accuracy with chance level, and watch an AI pose-estimation model try to build its own point-light display from a photograph.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 69 you watched a **point-light walker**: a dozen or so moving dots that instantly
resolve into a walking person, even though a single frozen frame looks like a random scatter.
In the advanced version, you tried identifying properties — gender, action, or emotion — from
the moving dots alone. This works because the brain's posterior superior temporal sulcus
(pSTS) has learned strong priors about how a jointed body moves, so it can reconstruct a whole
figure from a handful of points.

## Record your results

Watch several point-light clips (biomotionlab.ca or a "point-light emotion" search turns up
plenty) and guess before you check the answer. Log whether each guess was correct, and how
many alternatives you were choosing between.

In [ ]:
# example data: replace with yours -- guess before checking the answer for each clip
trials = pd.DataFrame({
    "clip": list(range(1, 9)),
    "task": ["gender"] * 4 + ["emotion"] * 4,
    "n_alternatives": [2, 2, 2, 2, 4, 4, 4, 4],
    "correct": [True, True, False, True, True, False, True, True],
})
display(trials)
acc_by_task = trials.groupby("task").correct.mean() * 100
acc_by_task.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3.5))
plt.ylabel("% correct"); plt.ylim(0, 100); plt.xticks(rotation=0); plt.show()

## Compare

Lab 69 reports that observers identify emotion from point-light displays at rates "well
above chance" (Dittrich et al., 1996), without giving an exact figure. Chance level itself is
not a citation, just arithmetic: 1 divided by the number of alternatives you were guessing
among.

In [ ]:
chance = {task: 100 / n for task, n in trials.groupby("task").n_alternatives.first().items()}
for task, acc in acc_by_task.items():
    print(f"{task.capitalize()} task: {acc:.0f}% correct vs {chance[task]:.0f}% expected by chance.")
    if acc > chance[task]:
        print("  Above chance -- consistent with Lab 69's claim that these judgements are")
        print("  'well above chance', even from a dozen moving dots and nothing else.")
    else:
        print("  At or below chance in this small sample -- try a few more clips before concluding anything.")

## The AI side

The AI equivalent of reading a body from dots is **pose estimation**: recovering the
positions of joints from an image. Below, a small, open pose-estimation model (MediaPipe
Pose) looks at an ordinary photograph and reduces it to exactly the kind of joint-only
display Johansson used — a machine-made point-light walker.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# A public-domain NASA portrait (astronaut Eileen Collins), bundled with scikit-image.
# It is a full photograph, not a point-light clip, because the model needs one (see "after"
# below). Only the upper body is visible, so the model can place only upper-body points.
from skimage import data as skdata
from PIL import Image as PILImage
PILImage.fromarray(skdata.astronaut()).save("photo.jpg")
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task

base_options = mp_python.BaseOptions(model_asset_path="pose_landmarker.task")
detector = mp_vision.PoseLandmarker.create_from_options(mp_vision.PoseLandmarkerOptions(base_options=base_options))

image = mp.Image.create_from_file("photo.jpg")
result = detector.detect(image)
photo = cv2.cvtColor(cv2.imread("photo.jpg"), cv2.COLOR_BGR2RGB)
h, w = photo.shape[:2]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(photo); axes[0].set_title("What MediaPipe was given"); axes[0].axis("off")
axes[1].set_facecolor("black")
if result.pose_landmarks:
    xs = [lm.x * w for lm in result.pose_landmarks[0]]
    ys = [lm.y * h for lm in result.pose_landmarks[0]]
    axes[1].scatter(xs, ys, c="white", s=20)
    axes[1].set_xlim(0, w); axes[1].set_ylim(h, 0)
    axes[1].set_title("Joints only: a machine-made point-light display")
else:
    axes[1].text(0.5, 0.5, "No pose detected", color="white", ha="center")
axes[1].axis("off")
plt.tight_layout(); plt.show()
n_found = len(result.pose_landmarks[0]) if result.pose_landmarks else 0
print(f"MediaPipe found {n_found} body landmarks in the photograph.")

Notice that MediaPipe needed a full, detailed photograph — colour, texture, lighting, a
recognisable human shape — to find these joints in the first place. If you tried running the
same code on an actual point-light clip instead of a photograph, it would very likely fail to
find a body at all: it was trained to recognise people by their photographic appearance
(skin, clothing, edges), not to reconstruct a figure from sparse, textureless motion the way
your pSTS does. That is the opposite of the human case: your brain needs almost no visual
detail, only the pattern of movement, to see a person.

## Pool the class data

One person's guesses are noisy. Pool the class's point-light identification accuracy to see
whether it holds up well above chance across many observers and many clips.

In [ ]:
import io
csv_text = """id,task,n_alternatives,correct
P01,gender,2,1
P02,gender,2,1
P03,gender,2,0
P04,emotion,4,1
P05,emotion,4,1
P06,emotion,4,0
P07,gender,2,1
P08,emotion,4,1
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
acc = class_df.groupby("task").correct.mean() * 100
boot = {}
for task, group in class_df.groupby("task"):
    vals = group.correct.values
    boot[task] = [np.random.choice(vals, len(vals), replace=True).mean() * 100 for _ in range(5000)]
for task in acc.index:
    lo, hi = np.percentile(boot[task], [2.5, 97.5])
    print(f"{task.capitalize()}: class accuracy {acc[task]:.0f}% (95% bootstrap interval {lo:.0f}-{hi:.0f}%)")
acc.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3.5))
plt.ylabel("% correct"); plt.ylim(0, 100); plt.show()

## Questions to think about

1. Your accuracy above depended on how many alternatives you were guessing among. If you replicated this with a friend using a 3-alternative emotion set instead of 4, what chance level would you now need to beat, and why does that matter for judging whether either of you is genuinely reading the motion?
2. The scrambled control in Figure 8.1(c) uses the same 13 dots as the walker, just placed at random positions. Why does scrambling the positions (while keeping the same dots) destroy the percept, when scrambling a photograph's pixels would arguably damage it far less severely?
3. MediaPipe needed a full photograph to find a body, while your brain needed only a dozen dots. What would a model need to add, beyond more layers or more training photographs, to close that gap?
4. Gait-recognition systems can identify individuals from how they walk, much as you might recognise a friend's walk from a distance. What does the existence of such systems suggest about how much individually distinctive information is actually present in an ordinary walking pattern?

## Challenge

Film a friend walking, then use any pose-estimation app or the MediaPipe code above (adapted to
video) to turn the footage into moving dots only. Show the point-light version to someone else
and see whether they can identify who it is, or how the person is feeling, exactly as in
Dittrich et al.'s original demonstrations.

In [ ]:
# Example data: replace with your own results from showing a point-light video (made from a
# friend's walking footage) to several people and asking them to guess who it is.
identification_trials = pd.DataFrame({
    "viewer": ["V1", "V2", "V3", "V4", "V5"],
    "correctly_identified_walker": [True, False, True, True, False],  # example data
})
pct_correct = identification_trials.correctly_identified_walker.mean() * 100
print(f"{pct_correct:.0f}% of viewers correctly identified the walker from point-light motion alone.")
print("Compare this with chance level for your group size (1 out of however many people they")
print("could have guessed), and with the accuracy you measured for yourself earlier in this notebook.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-69.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")